In [ ]:
import pandas as pd
import numpy as np

## 1. Import the Dataset

In [ ]:
PATH = "<CLEANED_DATA_DIR>/"
trip_stop_events = pd.read_csv(PATH+ "trip_stop_events.csv")
print(trip_stop_events.columns)
print(trip_stop_events.head(5))

## Removing Trips with No Route Information

In [ ]:
# Drop rows with empty route_id (Missing Type B) => Trips with stops that are not planned
# route_id= Null means missing type B, with no route records, but only exist in stop record
before_drop = trip_stop_events.shape[0]
trip_stop_events = trip_stop_events.dropna(subset=['route_id'])
after_drop = trip_stop_events.shape[0]

print(f"Due to empty route_id, dropped {before_drop - after_drop} rows (Missing Type B)")
print(f"Current trip_stop_events shape : {trip_stop_events.shape}")


## Compute Execution Rate per Trip

In [ ]:
# Aggregate stop counts per trip
trip_stats = (
    trip_stop_events.groupby(['trip_id', 'route_id'])
    .agg(
        total_stops=('completed', 'count'),
        completed_stops=('completed', 'sum')
    )
    .reset_index()
)

# Execution rate as a proportion
trip_stats['execution_rate'] = trip_stats['completed_stops'] / trip_stats['total_stops']

trip_stats.head()

## Assigning trip score

In [ ]:
trip_stats['score'] = (trip_stats['execution_rate'] * 100).round().astype(int)


## Summary Statistics

In [ ]:
summary = (trip_stats['score'].describe()
           .rename({
               'count': 'trips',
               'mean':  'avg_score', 
               'std':   'std_score',
               'min':   'min_score',
               '25%':   'p25_score',
               '50%':   'median_score',
               '75%':   'p75_score',
               'max':   'max_score'
           }))

print(summary)

## Assigning Trips into Different Tiers

In [ ]:
def assign_tier(score):
    if score >= 95:
        return 'Excellent (>94)'
    elif score >= 78:  # P25 threshold
        return 'Good (>77)'
    elif score >= 50:
        return 'Partial (>50)'
    else:
        return 'Poor (<50)'

trip_stats['tier'] = trip_stats['score'].apply(assign_tier)

In [ ]:
import matplotlib.pyplot as plt

# Count trips per tier
tier_counts = trip_stats['tier'].value_counts().reindex([
    'Excellent (>94)',
    'Good (>77)',
    'Partial (>50)',
    'Poor (<50)'
])

# Shades of red (light → dark)
colors = ['#ffcccc', '#ff9999', '#ff6666', '#cc0000']

plt.figure(figsize=(8, 5))

# Thinner bars → width=0.45 (adjust if you want even slimmer)
bars = plt.bar(tier_counts.index, tier_counts.values, 
               color=colors, width=0.45)

# Add labels on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2,
             height + 50,
             f"{height:,}",
             ha='center', fontsize=11)

plt.title("Performance Tier Distribution", fontsize=14, fontweight='bold')
plt.ylabel("Number of Trips")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
output_cols = ['trip_id', 'route_id', 'total_stops', 'completed_stops', 'execution_rate', 'score', 'tier']
trip_stats[output_cols].sort_values('score', ascending=False).to_excel(
    PATH+'route_execution_scores.xlsx', index=False
)
print(f"Exported {len(trip_stats)} trips → route_execution_scores.xlsx")